# Notebook 3 – Revenue Optimization

**Question**: What pricing strategies maintain revenue while managing demand?

**Approach**:
1. Describe actual revenue impact of the policy.
2. Estimate price elasticity of demand using OLS as an interpretable baseline.
3. Build a Random Forest and XGBoost demand model on bucket-level aggregated data.
4. Use XGBoost + SHAP to measure how `cbd_congestion_fee` drives demand across segments.
5. Simulate a revenue curve across a price grid using the XGBoost demand function.
6. Derive optimal pricing recommendations.

**Outline**
1. Load & Validate
2. Type Coercions
3. Revenue Accounting
4. PCA Feature Compression
5. Baseline: OLS Price Elasticity
6. Random Forest Demand Model
7. XGBoost Demand Model + SHAP
8. Elasticity by Segment (Partial Dependence)
9. Revenue Curve Simulation
10. Optimal Pricing Recommendations
11. Robustness Check
12. Conclusions

In [1]:
import gc
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd()))

from utils import (
    DATA_PATH,
    HOUR_BUCKET_ORDER,
    RANDOM_STATE,
    aggregate_volume,
    coerce_types,
    did_regression,
    load_and_validate,
)

np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')

## 1. Load & Validate

In [2]:
REQUIRED_COLS = [
    'pickup_hour_of_day', 'pickup_day_of_week', 'in_cbd_zone',
    'dataset_split', 'provider', 'fare_amount', 'total_amount',
    'cbd_congestion_fee', 'congestion_surcharge', 'airport_fee',
    'trip_distance', 'cost_per_mile',
    'temperature', 'precipitation', 'windspeed',
    'weather_clear', 'weather_cloudy', 'weather_rain',
]

df_raw = load_and_validate(DATA_PATH, REQUIRED_COLS)
df_raw.head(3)

Loaded 38,202,226 rows × 30 columns


,pickup_datetime,provider,dropoff_datetime,PULocationID,DOLocationID,trip_distance,trip_time,fare_amount,tax,tolls_amount,...,windspeed,weather_clear,weather_cloudy,weather_rain,weather_snow,dataset_split,pickup_day_of_week,pickup_hour_of_day,dropoff_day_of_week,dropoff_hour_of_day
0,2024-01-23 00:50:15,1,2024-01-23 01:21:53,132,33,26.62,1898,96.8,0.5,0.0,...,11.6,0,1,0,0.0,train,1,0,1,1
1,2024-01-14 20:21:20,1,2024-01-14 20:26:25,236,43,0.76,305,7.2,0.5,0.0,...,12.1,0,1,0,0.0,train,6,20,6,20
2,2024-01-23 00:23:16,1,2024-01-23 00:25:55,246,50,0.70,159,5.8,0.5,0.0,...,11.6,0,1,0,0.0,train,1,0,1,0


## 2. Type Coercions

In [ ]:
df = coerce_types(df_raw)

print(f"Pre trips : {(df['post'] == 0).sum():,}")
print(f"Post trips: {(df['post'] == 1).sum():,}")
print(f"\ncbd_congestion_fee stats:\n{df['cbd_congestion_fee'].describe().round(2)}")

# need to clean up for memory preservation
del df_raw
gc.collect()

## 3. Revenue Accounting

Compare total and per-trip revenue components pre vs. post.

In [ ]:
revenue_summary = (
    df.groupby(['post', 'in_cbd_zone'], observed=True)
    .agg(
        trip_count=('fare_amount', 'count'),
        total_fare=('fare_amount', 'sum'),
        total_revenue=('total_amount', 'sum'),
        total_cbd_fee=('cbd_congestion_fee', 'sum'),
        avg_fare=('fare_amount', 'mean'),
        avg_total=('total_amount', 'mean'),
        avg_cbd_fee=('cbd_congestion_fee', 'mean'),
    )
    .reset_index()
)
revenue_summary['period_label'] = revenue_summary['post'].map({0: 'Pre-Policy', 1: 'Post-Policy'})
revenue_summary['zone_label'] = revenue_summary['in_cbd_zone'].map(
    {True: 'CBD Zone', False: 'Non-CBD Zone'}
)

print("Revenue Summary:")
print(revenue_summary[[
    'period_label', 'zone_label', 'trip_count',
    'avg_fare', 'avg_total', 'avg_cbd_fee', 'total_cbd_fee'
]].to_string(index=False))

In [ ]:
rev_bucket = (
    df[df['in_cbd_zone']]
    .groupby(['post', 'hour_bucket', 'day_bucket'], observed=True)
    .agg(
        trip_count=('fare_amount', 'count'),
        total_cbd_fee=('cbd_congestion_fee', 'sum'),
        avg_cbd_fee=('cbd_congestion_fee', 'mean'),
    )
    .reset_index()
)
rev_bucket['period_label'] = rev_bucket['post'].map({0: 'Pre-Policy', 1: 'Post-Policy'})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(
    data=rev_bucket, x='hour_bucket', y='total_cbd_fee', hue='period_label',
    order=HOUR_BUCKET_ORDER, ax=axes[0], errorbar=None
)
axes[0].set_title('Total CBD Fee Revenue by Hour Bucket', fontsize=12)
axes[0].set_xlabel('Hour Bucket')
axes[0].set_ylabel('Total CBD Fee ($)')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(
    data=rev_bucket, x='hour_bucket', y='avg_cbd_fee', hue='period_label',
    order=HOUR_BUCKET_ORDER, ax=axes[1], errorbar=None
)
axes[1].set_title('Avg CBD Fee per Trip by Hour Bucket', fontsize=12)
axes[1].set_xlabel('Hour Bucket')
axes[1].set_ylabel('Avg CBD Fee ($)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('CBD Zone Revenue – Pre vs. Post Policy', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. PCA Feature Compression

The feature matrix for ML models includes weather, time-bucket dummies, and provider dummies.
PCA compresses these into a smaller set of orthogonal components before training.

In [ ]:
# Build bucket-level volume table (provider dimension added for segment analysis)
vol = aggregate_volume(
    df,
    groupby_cols=[
        'post', 'in_cbd_zone', 'hour_bucket', 'day_bucket',
        'pickup_hour_of_day', 'pickup_day_of_week',
    ],
)

# clean up again here
del df
gc.collect()
print(f"Volume table shape: {vol.shape}")
vol.head(5)

In [ ]:
# One-hot encode categorical columns for the feature matrix
vol_enc = pd.get_dummies(
    vol,
    columns=['hour_bucket', 'day_bucket'],
    drop_first=False
)

# Select numeric features for PCA
pca_input_cols = [
    c for c in vol_enc.columns
    if c.startswith(('hour_bucket_', 'day_bucket_'))
    or c in ['avg_temp', 'avg_precip', 'avg_wind', 'post', 'in_cbd_zone_int']
]

pca_input = vol_enc[pca_input_cols].fillna(0).astype(float)

scaler = StandardScaler()
pca_input_scaled = scaler.fit_transform(pca_input)

N_PCA = min(10, pca_input.shape[1] - 1)
pca = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
pca_components = pca.fit_transform(pca_input_scaled)

pca_df = pd.DataFrame(
    pca_components,
    columns=[f'pca_{i+1}' for i in range(N_PCA)],
    index=vol_enc.index
)

evr = pca.explained_variance_ratio_
print(f"PCA variance explained by {N_PCA} components: {evr.sum()*100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, N_PCA + 1), evr * 100)
ax.plot(range(1, N_PCA + 1), np.cumsum(evr) * 100, 'r-o', markersize=4, label='Cumulative')
ax.set_xlabel('Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Feature PCA – Explained Variance')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Assemble final ML feature matrix
ml_df = pd.concat([
    vol_enc[['log_trip_count', 'trip_count', 'avg_cbd_fee', 'post', 'in_cbd_zone_int',
             'avg_temp', 'avg_precip', 'avg_wind', 'pickup_hour_of_day', 'pickup_day_of_week']].reset_index(drop=True),
    pca_df.reset_index(drop=True),
], axis=1)

ml_df = ml_df.dropna(subset=['log_trip_count', 'avg_cbd_fee'])

PRICE_COL = 'avg_cbd_fee'
TARGET_COL = 'log_trip_count'

FEATURE_COLS = (
    [PRICE_COL, 'post', 'in_cbd_zone_int', 'avg_temp', 'avg_precip', 'avg_wind',
     'pickup_hour_of_day', 'pickup_day_of_week']
    + [f'pca_{i+1}' for i in range(N_PCA)]
)

X_ml = ml_df[FEATURE_COLS].astype(float)
y_ml = ml_df[TARGET_COL].astype(float)

print(f"ML feature matrix: {X_ml.shape}")

## 5. Baseline: OLS Price Elasticity

Log-log OLS regression gives a direct, interpretable elasticity estimate:
`log(volume) ~ log(fee + 1) + controls`. The coefficient on `log(fee + 1)` is the
price elasticity of demand (PED).

In [ ]:
ml_df['log_fee'] = np.log1p(ml_df['avg_cbd_fee'])

ols_formula = (
    'log_trip_count ~ log_fee + post + in_cbd_zone_int '
    '+ C(pickup_hour_of_day) + C(pickup_day_of_week) '
    '+ avg_temp + avg_precip + avg_wind'
)
ols_model = smf.ols(ols_formula, data=ml_df).fit(cov_type='HC3')
print(ols_model.summary())

In [ ]:
ped_coef = ols_model.params.get('log_fee', np.nan)
ped_ci = ols_model.conf_int().loc['log_fee'] if 'log_fee' in ols_model.params else [np.nan, np.nan]

print("OLS Price Elasticity of Demand:")
print(f"  Elasticity (log-log coef): {ped_coef:.4f}")
print(f"  95% CI                   : [{ped_ci[0]:.4f}, {ped_ci[1]:.4f}]")
print(f"  Interpretation: 1% increase in fee → {ped_coef:.2f}% change in trip volume")

## 6. Random Forest Demand Model

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_cv_rmse = np.sqrt(-cross_val_score(
    rf_reg, X_ml, y_ml, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1
))
rf_cv_r2 = cross_val_score(rf_reg, X_ml, y_ml, cv=kf, scoring='r2', n_jobs=-1)

print(f"RF  5-fold CV RMSE : {rf_cv_rmse.mean():.4f} ± {rf_cv_rmse.std():.4f}")
print(f"RF  5-fold CV R²   : {rf_cv_r2.mean():.4f} ± {rf_cv_r2.std():.4f}")

rf_reg.fit(X_ml, y_ml)

In [ ]:
rf_imp = (
    pd.DataFrame({'feature': FEATURE_COLS, 'importance': rf_reg.feature_importances_})
    .sort_values('importance', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=rf_imp.head(15), x='importance', y='feature', ax=ax, palette='Blues_r')
ax.set_title('RF Feature Importance – Demand Model', fontsize=13)
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

## 7. XGBoost Demand Model + SHAP

XGBoost provides higher predictive accuracy on tabular bucket data.
SHAP values show how each feature — especially `avg_cbd_fee` — drives predicted demand
across different bucket types.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)

xgb_cv_rmse = np.sqrt(-cross_val_score(
    xgb_model, X_ml, y_ml, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1
))
xgb_cv_r2 = cross_val_score(xgb_model, X_ml, y_ml, cv=kf, scoring='r2', n_jobs=-1)

print(f"XGB 5-fold CV RMSE: {xgb_cv_rmse.mean():.4f} ± {xgb_cv_rmse.std():.4f}")
print(f"XGB 5-fold CV R²  : {xgb_cv_r2.mean():.4f} ± {xgb_cv_r2.std():.4f}")
print(f"\nModel comparison:")
print(f"  OLS  R² (train): {r2_score(y_ml, ols_model.fittedvalues):.4f}")
print(f"  RF   R² (5-fold): {rf_cv_r2.mean():.4f}")
print(f"  XGB  R² (5-fold): {xgb_cv_r2.mean():.4f}")

xgb_model.fit(X_ml, y_ml)

In [ ]:
try:
    import shap

    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_ml)

    # Summary plot: overall feature importance via SHAP
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values, X_ml,
        feature_names=FEATURE_COLS,
        plot_type='bar',
        show=False
    )
    plt.title('XGBoost SHAP Feature Importance (Mean |SHAP|)', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Beeswarm: how fee level affects demand
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_ml, feature_names=FEATURE_COLS, show=False)
    plt.title('XGBoost SHAP Value Distribution', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Dependence plot: cbd_fee vs SHAP
    fee_idx = FEATURE_COLS.index(PRICE_COL)
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(
        fee_idx, shap_values, X_ml,
        feature_names=FEATURE_COLS,
        show=False
    )
    plt.title(f'SHAP Dependence: {PRICE_COL} → Demand', fontsize=13)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("shap not installed – skipping SHAP analysis. Run: pip install shap")

## 8. Elasticity by Segment (Partial Dependence)

Compute partial dependence of `avg_cbd_fee` on `log_trip_count` using the XGBoost model,
separately for key segments: peak vs. off-peak, and CBD vs. non-CBD.

In [ ]:
price_grid = np.linspace(0, 30, 60)

RUSH_HOURS = {7, 8, 9, 16, 17, 18, 19}
SEGMENTS = {
    'Morning/Evening Rush (CBD)': (X_ml['pickup_hour_of_day'].isin(RUSH_HOURS)) & (X_ml['in_cbd_zone_int'] == 1),
    'Off-Peak (CBD)'            : (~X_ml['pickup_hour_of_day'].isin(RUSH_HOURS)) & (X_ml['in_cbd_zone_int'] == 1),
    'Non-CBD'                   : X_ml['in_cbd_zone_int'] == 0,
}

fig, ax = plt.subplots(figsize=(10, 6))

for seg_label, seg_mask in SEGMENTS.items():
    X_seg = X_ml[seg_mask]
    if len(X_seg) < 5:
        print(f"Segment '{seg_label}' has too few rows, skipping.")
        continue

    # For each price point, replace avg_cbd_fee with the counterfactual price and predict
    pdp_preds = []
    for p in price_grid:
        X_cf = X_seg.copy()
        X_cf[PRICE_COL] = p
        pdp_preds.append(xgb_model.predict(X_cf).mean())

    ax.plot(price_grid, pdp_preds, label=seg_label, linewidth=2)

ax.set_xlabel('CBD Congestion Fee ($)')
ax.set_ylabel('Predicted Log Trip Volume (mean)')
ax.set_title('Partial Dependence of Congestion Fee on Demand by Segment', fontsize=13)
ax.legend(title='Segment')
plt.tight_layout()
plt.show()

In [ ]:
# Compute implied arc elasticity between current fee (~$9) and baseline ($0) from PDP curves
baseline_price = 0.5  # small positive to avoid log(0)
current_price = 9.0

print("Implied Arc Elasticities by Segment:")
for seg_label, seg_mask in SEGMENTS.items():
    X_seg = X_ml[seg_mask]
    if len(X_seg) < 5:
        continue
    X_base = X_seg.copy(); X_base[PRICE_COL] = baseline_price
    X_curr = X_seg.copy(); X_curr[PRICE_COL] = current_price
    vol_base = np.expm1(xgb_model.predict(X_base).mean())
    vol_curr = np.expm1(xgb_model.predict(X_curr).mean())
    if vol_base > 0:
        elasticity = (np.log(vol_curr) - np.log(vol_base)) / (np.log(current_price) - np.log(baseline_price))
        pct_chg = (vol_curr - vol_base) / vol_base * 100
        print(f"  {seg_label}: elasticity={elasticity:.3f}, demand change={pct_chg:+.1f}%")

## 9. Revenue Curve Simulation

Sweep `cbd_congestion_fee` from $0 to $30 and compute:
`Revenue(p) = p × predicted_demand(p)` for each bucket, then aggregate.

In [ ]:
# Simulate over CBD-zone rows in the post period (most policy-relevant)
X_sim_base = X_ml[(X_ml['in_cbd_zone_int'] == 1) & (X_ml['post'] == 1)].copy()

if len(X_sim_base) == 0:
    print("No post-policy CBD rows found; using all CBD rows for simulation.")
    X_sim_base = X_ml[X_ml['in_cbd_zone_int'] == 1].copy()

price_grid_fine = np.arange(0, 31, 1.0)
revenue_results = []

for p in price_grid_fine:
    X_cf = X_sim_base.copy()
    X_cf[PRICE_COL] = p
    pred_log_vol = xgb_model.predict(X_cf)
    pred_vol = np.expm1(pred_log_vol).sum()   # total trips at this price point
    revenue = p * pred_vol
    revenue_results.append({'price': p, 'predicted_volume': pred_vol, 'revenue': revenue})

rev_df = pd.DataFrame(revenue_results)

# Normalize to baseline (price=0) for relative comparison
baseline_vol = rev_df.loc[rev_df['price'] == 0, 'predicted_volume'].values[0]
rev_df['volume_pct_of_baseline'] = rev_df['predicted_volume'] / baseline_vol * 100

rev_df.head()

In [ ]:
opt_row = rev_df.loc[rev_df['revenue'].idxmax()]
demand_target = rev_df.loc[(rev_df['volume_pct_of_baseline'] <= 90), 'price'].min()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Revenue curve
axes[0].plot(rev_df['price'], rev_df['revenue'], 'steelblue', linewidth=2)
axes[0].axvline(opt_row['price'], color='green', linestyle='--', label=f'Revenue max (${opt_row["price"]:.0f})')
if not np.isnan(demand_target):
    axes[0].axvline(demand_target, color='orange', linestyle='--', label=f'≥10% demand drop (${demand_target:.0f})')
axes[0].set_xlabel('CBD Congestion Fee ($)')
axes[0].set_ylabel('Simulated Total Revenue ($)')
axes[0].set_title('Revenue Curve vs. Congestion Fee', fontsize=13)
axes[0].legend()

# Demand curve
axes[1].plot(rev_df['price'], rev_df['volume_pct_of_baseline'], 'firebrick', linewidth=2)
axes[1].axhline(90, color='orange', linestyle='--', label='10% demand reduction threshold')
axes[1].axvline(opt_row['price'], color='green', linestyle='--', label=f'Revenue-maximizing price (${opt_row["price"]:.0f})')
axes[1].set_xlabel('CBD Congestion Fee ($)')
axes[1].set_ylabel('Trip Volume (% of no-fee baseline)')
axes[1].set_title('Demand Curve vs. Congestion Fee', fontsize=13)
axes[1].legend()

plt.suptitle('XGBoost Revenue & Demand Simulation', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f"Revenue-maximizing fee : ${opt_row['price']:.0f}")
print(f"  → Predicted volume   : {opt_row['predicted_volume']:,.0f} trips")
print(f"  → Predicted revenue  : ${opt_row['revenue']:,.0f}")
if not np.isnan(demand_target):
    demand_row = rev_df[rev_df['price'] == demand_target].iloc[0]
    print(f"Fee for ≥10% demand drop: ${demand_target:.0f}")
    print(f"  → Predicted volume  : {demand_row['predicted_volume']:,.0f} trips")
    print(f"  → Predicted revenue : ${demand_row['revenue']:,.0f}")

## 10. Optimal Pricing Recommendations

In [ ]:
current_fee = 9.0
current_row = rev_df.iloc[(rev_df['price'] - current_fee).abs().argsort().iloc[0]]

scenarios = [
    ('No fee (baseline)',          rev_df[rev_df['price'] == 0].iloc[0]),
    ('Current policy ($9)',        current_row),
    ('Revenue-maximizing',         rev_df.loc[rev_df['revenue'].idxmax()]),
]

# Add demand-reduction scenario if it exists
if not np.isnan(demand_target):
    dr_row = rev_df[rev_df['price'] == demand_target]
    if len(dr_row) > 0:
        scenarios.append(('≥10% Demand Reduction', dr_row.iloc[0]))

scenario_table = pd.DataFrame([
    {
        'Scenario'           : name,
        'Fee ($)'            : f"{row['price']:.0f}",
        'Pred. Trips'        : f"{row['predicted_volume']:,.0f}",
        'Vol vs. Baseline %' : f"{row['volume_pct_of_baseline']:.1f}%",
        'Pred. Revenue ($)'  : f"{row['revenue']:,.0f}",
    }
    for name, row in scenarios
])

print("Pricing Scenario Outcomes:")
print(scenario_table.to_string(index=False))

## 11. Robustness Check

Re-run XGBoost without PCA compression (using raw features) to verify that the
key elasticity finding is stable.

In [ ]:
RAW_FEATURE_COLS = [
    PRICE_COL, 'post', 'in_cbd_zone_int',
    'avg_temp', 'avg_precip', 'avg_wind',
    'pickup_hour_of_day', 'pickup_day_of_week',
]

X_raw = ml_df[RAW_FEATURE_COLS].astype(float)

xgb_robust = xgb.XGBRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
)

r2_raw = cross_val_score(xgb_robust, X_raw, y_ml, cv=kf, scoring='r2', n_jobs=-1)
r2_pca = xgb_cv_r2  # already computed above

print("Robustness Check – XGBoost R² (5-fold CV):")
print(f"  With PCA features    : {r2_pca.mean():.4f} ± {r2_pca.std():.4f}")
print(f"  Without PCA (raw)    : {r2_raw.mean():.4f} ± {r2_raw.std():.4f}")

# Check revenue curve stability with raw-feature model
xgb_robust.fit(X_raw, y_ml)
X_sim_raw = X_sim_base[RAW_FEATURE_COLS].copy()

rev_robust = []
for p in price_grid_fine:
    X_cf = X_sim_raw.copy()
    X_cf[PRICE_COL] = p
    pred_vol = np.expm1(xgb_robust.predict(X_cf)).sum()
    rev_robust.append({'price': p, 'revenue': p * pred_vol})
rev_robust_df = pd.DataFrame(rev_robust)

opt_robust = rev_robust_df.loc[rev_robust_df['revenue'].idxmax()]
print(f"\nRevenue-maximizing fee (without PCA): ${opt_robust['price']:.0f}")
print(f"Revenue-maximizing fee (with PCA)   : ${opt_row['price']:.0f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rev_df['price'], rev_df['revenue'], label='With PCA', linewidth=2)
ax.plot(rev_robust_df['price'], rev_robust_df['revenue'], '--', label='Without PCA (raw)', linewidth=2)
ax.set_xlabel('CBD Congestion Fee ($)')
ax.set_ylabel('Simulated Revenue ($)')
ax.set_title('Revenue Curve Robustness Check', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 12. Conclusions

- **Revenue Accounting**: Compares actual total and per-trip revenue components pre vs. post policy, broken down by bucket.
- **OLS Baseline**: Provides a direct, interpretable price elasticity of demand estimate with confidence intervals.
- **Random Forest**: Captures nonlinear demand relationships and ranks features by importance.
- **XGBoost + SHAP**: Most accurate demand model; SHAP values reveal exactly how the congestion fee shifts demand across rush-hour vs. off-peak and CBD vs. non-CBD segments.
- **Revenue Curve**: Identifies the revenue-maximizing fee and the minimum fee that achieves a ≥10% demand reduction target.
- **Robustness Check**: Confirms that the revenue-maximizing fee estimate is stable with and without PCA feature compression.

In [ ]:
# cleanup
import gc
gc.collect()